# 共有Hadoop配布物の更新 — ホスト単位

NameNodeの更新後、Hadoopのcurrentをホストごとに切り替える。HDFS・YARNとクライアントはこの配布物を共有する。対象ごとに作業コピーを作る。

## *Operation Note*

*This is a cell for your own recording. ここに経緯を記述*

# 操作対象クラスタの設定

D10で作成したinventoryとgroup_varsを使用する。

In [ ]:
target_group = 'hadoop_all_testcluster'
target_group

In [ ]:
!ansible {target_group} -m ping

In [ ]:
import yaml
with open('group_vars/' + target_group) as configuration:
    kerberos_enabled = 'kerberos_realm' in yaml.safe_load(configuration)
kerberos_enabled

# 管理操作の認証

Kerberos構成の場合、HDFSの管理操作を行うホストでhdfsユーザーのkeytabを使って認証する。

In [ ]:
if kerberos_enabled:
    !ansible 'hadoop_namenode:hadoop_client' -l {target_group} --become --become-user hdfs -m shell -a 'kinit -kt /etc/hadoop/conf/hdfs.keytab hdfs/$(hostname -f)'

# Playbookの準備

In [ ]:
import os
import tempfile
work_dir = tempfile.mkdtemp()
work_dir

使用するNotebookと同じ版のPlaybookを取得する。

In [ ]:
!git clone https://github.com/NII-cloud-operation/Literate-computing-Hadoop.git {work_dir}/hadoop
!git -C {work_dir}/hadoop log -1

In [ ]:
playbook_dir = os.path.join(work_dir, 'hadoop/playbooks')
playbook_dir

In [ ]:
upgrade_host = 'XXX.XXX.XXX.114'
datanode_ipc_port = 9867
(upgrade_host, datanode_ipc_port)

In [ ]:
!ansible {upgrade_host} -a "hostname -f" -l {target_group}

In [ ]:
!ansible {upgrade_host} -m debug -a var=group_names -l {target_group}

inventoryの役割に従って対象ホストのサービスを操作する。クライアント専用ホストも更新対象に含める。

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs dfsadmin -report' -l {target_group}

Live datanodesが対象の全台であり、Missing blocks・Blocks with corrupt replicas・Under replicated blocksがすべて0であることを確認する。回復中なら時間をおいて再確認する。

In [ ]:
!ansible hadoop_client -a "yarn application -list -appStates SUBMITTED,ACCEPTED,RUNNING" -l {target_group}

実行中ジョブが0件であり、HBaseと履歴サービスが停止済みであることを確認する。

# YARNとZKFCの停止

In [ ]:
!ansible-playbook {playbook_dir}/stop_nodemanager.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/stop_resourcemanager.yml -l {upgrade_host}

In [ ]:
!ansible 'hadoop_namenode:&{upgrade_host}:&{target_group}' --become -m service -a "name=hadoop-hdfs-zkfc state=stopped"

# DataNodeの停止

このホストにDataNodeがある場合だけ実施する。Kerberos構成では対象DataNodeのprincipalで認証する。

In [ ]:
if kerberos_enabled:
    !ansible 'hadoop_slavenode:&{upgrade_host}:&{target_group}' --become --become-user hdfs -m shell -a 'kinit -kt /etc/hadoop/conf/hdfs.keytab hdfs/$(hostname -f)'

In [ ]:
!ansible 'hadoop_slavenode:&{upgrade_host}:&{target_group}' --become --become-user hdfs -m shell -a 'hdfs dfsadmin -shutdownDatanode $(hostname -f):{datanode_ipc_port} upgrade'

In [ ]:
!ansible 'hadoop_slavenode:&{upgrade_host}:&{target_group}' -m wait_for -a 'host={upgrade_host} port={datanode_ipc_port} state=stopped timeout=120'

# JournalNodeの停止

他のJournalNodeが過半数稼働していることを確認し、この1台だけを停止する。

In [ ]:
!ansible-playbook {playbook_dir}/stop_journalnode.yml -l {upgrade_host}

# 配布物の切替

In [ ]:
!ansible-playbook -CDv {playbook_dir}/switch_upgrade_package.yml --tags hadoop,jsvc -l {upgrade_host}

対象1台のHadoop currentとjsvc実行ファイルだけが変わることを確認する。

In [ ]:
!ansible-playbook {playbook_dir}/switch_upgrade_package.yml --tags hadoop,jsvc -l {upgrade_host}

In [ ]:
!ansible {upgrade_host} -a "readlink -f /opt/hadoop/current" -l {target_group}

In [ ]:
!ansible {upgrade_host} -a "hadoop version" -l {target_group}

# サービスの起動

In [ ]:
!ansible-playbook {playbook_dir}/start_journalnode.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/start_datanode.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/start_resourcemanager.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/start_nodemanager.yml -l {upgrade_host}

In [ ]:
!ansible 'hadoop_namenode:&{upgrade_host}:&{target_group}' --become -m service -a 'name=hadoop-hdfs-zkfc state=started'

In [ ]:
!ansible 'hadoop_slavenode:&{upgrade_host}:&{target_group}' -m wait_for -a 'host={upgrade_host} port={datanode_ipc_port} timeout=120'

In [ ]:
!ansible 'hadoop_slavenode:&{upgrade_host}:&{target_group}' --become --become-user hdfs -m shell -a 'hdfs dfsadmin -getDatanodeInfo $(hostname -f):{datanode_ipc_port}'

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs dfsadmin -report' -l {target_group}

Live datanodesが対象の全台であり、Missing blocks・Blocks with corrupt replicas・Under replicated blocksがすべて0であることを確認する。回復中なら時間をおいて再確認する。

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs haadmin -getAllServiceState' -l {target_group}

対象ホストのサービスが復帰したことを確認して次のホストへ進む。全ホストが完了したら、[HBase・Sparkの更新](O12i_Hadoop%20-%20Upgrade%20HBase%20and%20Spark.ipynb)へ進む。